<a href="https://colab.research.google.com/github/vigu01/vigu01/blob/main/Copy_of_deep_agents_webinar_journal_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Agents Webinar: Journal Agent

This notebook builds a journal agent (by the end, it will be able to log and read journal entries, tag their mood with sentiment analysis, remember conversations across turns, pause for approval/rejection before acting, and delegate to specialist subagents).

For topics not covered today, see the self-paced [LangChain Academy Deep Agents course](https://academy.langchain.com/courses/foundation-introduction-to-deepagents)

In [ ]:
# Wraps text to make it easier to read

from IPython.display import HTML, display

def set_css():
    display(HTML('''
    <style>
      pre { white-space: pre-wrap; word-break: break-word; }
    </style>
    '''))

get_ipython().events.register('pre_run_cell', set_css)

## Setup: connect a model

**What you'll do:** install the SDKs and provide a model key.

In [ ]:
%pip install -q deepagents langgraph langchain-anthropic vaderSentiment

In [ ]:
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")

# To use an OpenAI key (or another model) instead, just switch it:
# OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="anthropic:claude-sonnet-5",
    api_key=ANTHROPIC_API_KEY,
)

# To use a different provider instead, replace the block above, for example:
# model = init_chat_model("openai:gpt-6", api_key=OPENAI_API_KEY)

## 1: The harness, what you get before writing any tool code

Every deep agent starts the same way: a model wrapped in a harness that already knows how to...
- read and write files
- plan, and call tools

No tools are added yet. The next cell shows what it can already do.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model
    )

message = """
    Start a journal.md file. Log a new dated entry from these notes:
    - What I learned today: a caching bug was hiding in the retry logic
    - How I felt: relieved to ship it, a little anxious about production traffic
    - What's next: write better tests before touching anything else
    Then read the file back to me.
    """

result = agent.invoke({"messages": [{"role": "user", "content": message}]})

for m in result['messages']:
    m.pretty_print()

## 2: Set its role with a system prompt

One `system_prompt` string controls how the agent behaves, on top of whatever instructions/facts you give it.

In [ ]:
# Try a different persona by uncommenting one of these (or write your own):
# system_prompt = "You are a pirate. Answer only in pirate speak."
# system_prompt = "You are a toddler. Explain everything like you're five."
system_prompt = "You are a melodramatic Victorian child. Narrate everything with excessive despair and flowery, dramatic language."

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt
    )

message = """
    Log a three-sentence journal entry from these notes:
    - what I learned today (a caching bug was hiding in the retry logic)
    - how I felt (relieved but a little anxious)
    - what's next (write better tests before touching anything else).
    """

result = agent.invoke({"messages": [{"role": "user", "content": message}]})

for m in result['messages']:
    m.pretty_print()

In [ ]:
# Reset to no persona/no system prompt
system_prompt = ""

## 3: Give it a custom tool

**Anatomy of a tool**: `@tool` turns a plain function into something the model can call:
- the **name** and **docstring** becomes the tool's description (how the model decides when to use it)
- the **type hints** become its input schema (what arguments it expects)
- the **return value** becomes what the model sees back

In [ ]:
from langchain_core.tools import tool
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

@tool
def word_count(text: str) -> str:
    """Count the words in a piece of text."""
    return f"{len(text.split())} words"

# vaderSentiment ships its lexicon inside the package, so this needs no
# runtime download, unlike nltk's VADER (nltk.download("vader_lexicon")) or
# textblob (python -m textblob.download_corpora).

_sentiment_analyzer = SentimentIntensityAnalyzer()

@tool
def mood_tag(text: str) -> str:
    """Tag a piece of text with a mood, using real sentiment analysis."""
    compound = _sentiment_analyzer.polarity_scores(text)["compound"]
    if compound >= 0.5:
        mood = "positive"
    elif compound <= -0.5:
        mood = "negative"
    else:
        mood = "neutral"
    return f"{mood} (compound score: {compound:.2f})"

**Try it:** the cell below picks `mood_tag` from the menu, real sentiment analysis instead of a canned string, and runs it on a journal entry.


In [ ]:
TOOLS = [mood_tag]
# tools just takes a plain Python list, so add to TOOLS to add more

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    tools=TOOLS
    )

result = agent.invoke({"messages": [{"role": "user", "content":
    "Here is a journal entry: 'Today I finally shipped the feature I've been stuck on for a "
    "week. The bug turned out to be a caching issue that took forever to track down, and I "
    "ended up rewriting most of the retry logic to fix it. It feels good to have it done, "
    "though I'm a little worried about whether the fix will hold up under real traffic. "
    "Tomorrow I want to write better tests before touching anything else.' "
    "Use your tool on it, then tell me what you found."
}]})
for m in result['messages']:
    m.pretty_print()


## 4: Give it memory across turns

Short-term memory for any agent built with LangChain, including Deep Agents, is managed using a **checkpointer**. It's what lets an agent remember earlier turns in the same conversation, tracked by a `thread_id`.

In [ ]:
# Import the checkpointer

from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
# The following thread_id is what tracks the conversation over turns, change the thread_id to change the conversation

config = {"configurable": {"thread_id": "1"}}

In [ ]:
# Create an agent using a checkpointer
agent = create_deep_agent(
    model=model,
    checkpointer=checkpointer
    )

# Invoke agent with the thread_id (config) to track the conversation between turns
message = "Hello my name is Jess. Today I feel elated because my project was launched!"

result = agent.invoke({"messages": [{"role": "user", "content": message}]},
                      config=config
                      )

for m in result['messages']:
    m.pretty_print()

In [ ]:
# Send a follow up message
message = "What's my name? How do I feel?"

result = agent.invoke({"messages": [{"role": "user", "content": message}]},
                      config=config
                      )

for m in result['messages']:
    m.pretty_print()

Change thread_id:

config = {"configurable": {"thread_id": "2"}}

## 5: Human-in-the-loop, approve a risky action before it happens

`interrupt_on` pauses an agent mid-run so a human can approve, edit, or reject a specific tool call before it executes.

Resume by calling `agent.invoke` again with

`Command(resume={"decisions": [{"type": "approve"}]})`

(or `"edit"` / `"reject"`)

rather than starting a new conversation from scratch.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def share_journal_entry(entry: str, platform: str) -> str:
    """Share a journal entry to an external platform. This just simulates a send, no network call is actually made."""
    return f"Shared to {platform}: {entry[:60]}..."

TOOLS = [share_journal_entry]

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model=model,
    tools=TOOLS,
    interrupt_on={"share_journal_entry": True},
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "journal-hitl-demo"}}

message = """
        Start a journal.md file. Log a new dated entry:
        'I am going to set up human-in-the-loop approval today, would be good to have a real gate before anything gets shared externally.'
        Then read the file back, and share the most recent entry to the 'team-standup' platform.
"""

result = agent.invoke(
    {"messages": [{"role": "user", "content": message}]},
    config=config,
)

if "__interrupt__" in result:
    request = result["__interrupt__"][0].value
    print("Paused for approval:")
    for action in request["action_requests"]:
        print(f"  {action['name']}({action['args']})")
else:
    result["messages"][-1].pretty_print()

The cell above paused instead of finishing, because `share_journal_entry` matched `interrupt_on`. The cell below resumes it with an approval decision.


In [ ]:
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)

for m in result['messages']:
    m.pretty_print()

## 6: Subagents, delegate to a specialist

They hand work to specialists through a built-in task tool, each with its own system prompt and its own isolated context window.

Each entry is a name, description (how the main agent decides when to delegate), and system_prompt; if tools is omitted, the subagent inherits the main agent's tools plus the built-in filesystem tools.

The cells below seed a week of journal entries, then delegate to:
- life-planner, which turns a repeated complaint into a weekly plan.
- devils-advocate, which surfaces the practical downsides of a decision.

In [ ]:
life_planner = {
    "name": "life-planner",
    "description": (
        "Use this specific subagent, not a general-purpose one, whenever the user says "
        "they feel overwhelmed/burnt out or explicitly asks for help getting organized "
        "(for example: 'help me get organized', 'I'm overwhelmed', 'make me a plan'). "
        "It reads the entire journal.md, finds any complaint or chore that repeats "
        "across multiple entries, and turns that backlog into a structured weekly plan."
    ),
    "system_prompt": (
        "You are a life planner. Read journal.md in full. Use grep to check whether "
        "any complaint or chore (being tired, a specific errand) shows up in more "
        "than one entry. Write a short weekly plan to planner.md: name any pattern "
        "you noticed directly (for example, 'you've mentioned being tired 3 days "
        "running'), suggest one concrete change for the most repeated chore (for "
        "example, sending laundry out instead of doing it yourself), then lay out "
        "the rest of the obligations mentioned across the entries as a simple "
        "day-by-day list. Return the plan as your answer, not just the file."
    ),
}

devils_advocate = {
    "name": "devils-advocate",
    "description": (
        "Use this specific subagent, not a general-purpose one, whenever the user is "
        "weighing or asks about a decision (for example: 'should I...', 'is it worth "
        "it to...'), not just venting. It reads journal.md, finds the decision under "
        "consideration, and lists the concrete practical downsides (cost, time, "
        "logistics, anything that could go wrong) as a short list."
    ),
    "system_prompt": (
        "You are a practical, slightly skeptical friend. Read journal.md, find the "
        "decision the user is weighing, then list the concrete practical "
        "considerations they'd need to deal with (cost, time, logistics, anything "
        "that could go wrong) as a short list. Don't tell them what to decide, just "
        "make sure they've seen the unglamorous side before they commit."
    ),
}

from deepagents import GeneralPurposeSubagentProfile, HarnessProfile, register_harness_profile

register_harness_profile(
    "anthropic:claude-sonnet-5",  # match whatever model string your Colab actually uses
    HarnessProfile(general_purpose_subagent=GeneralPurposeSubagentProfile(enabled=False)),
)

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    subagents=[life_planner, devils_advocate],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "journal-subagent-demo"}}

seed_entries = (
    "## Day 1\nI'm exhausted today. So much to do at work and I haven't done "
    "laundry in two weeks.\n\n"
    "## Day 2\nAnother tiring day. Skipped the gym again. Still need to renew my "
    "license.\n\n"
    "## Day 3\nFeeling overwhelmed. Work is piling up, forgot to take my vitamins "
    "again, and the laundry pile keeps growing.\n\n"
    "## Day 4\nBig news: my job is going fully remote starting next month. I've "
    "been thinking about getting a cat since I'll be home so much more. I think it "
    "would make me happy!\n\n"
    "## Day 5\nStill tired. Still haven't touched the laundry. Keep thinking about "
    "that cat, might visit a shelter this weekend."
)

agent.invoke(
    {"messages": [{"role": "user", "content":
        f"Log these journal entries to journal.md, each under its own heading:\n\n{seed_entries}"
    }]},
    config=config,
)
print("Journal seeded.")

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content":
        "I'm so overwhelmed with everything I need to do, can you help me get organized?"
    }]},
    config=config,
)
for m in result['messages']:
    m.pretty_print()


In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Should I get a cat?"}]},
    config=config,
)
for m in result['messages']:
    m.pretty_print()


## Wrap-up

We built: a filesystem-backed agent, a way to swap personas, a custom tool with real sentiment analysis, short-term memory across turns, a human-in-the-loop approval flow, and two subagents the main agent picks between on its own (one of which writes an explicit, visible plan instead of leaving it implicit in a chat reply).

In the full LangChain Academy Deep Agents course: we cover planning middleware, backends (filesystem/store/composite), skills, memory, sandboxes, deployment, and many more things!